# Task 1: Conceptual Understanding

---

## Q1. What is the difference between "Love" and "love" in NLP?

In NLP, "Love" and "love" are treated as different words because most systems are case-sensitive.

"Love" (with a capital L) is usually used at the beginning of a sentence, while "love" (lowercase) is the standard form used in analysis.

This can create a problem because the same word is counted twice, increasing the vocabulary size unnecessarily. To avoid this, we convert all text to lowercase during preprocessing. This helps in maintaining consistency and improving model performance.

---

## Q2. What happens if stopwords are not removed?

Stopwords are very common words like "is", "the", and "and" that do not add much meaning to the text.

If stopwords are not removed:
- They dominate word frequency counts
- They add noise to the data
- They increase the size of the dataset
- They make models less efficient and less accurate

Removing stopwords helps focus on meaningful words and improves overall performance.

---

## Q3. Two real-world scenarios where removing stopwords can be harmful

1. Sentiment Analysis with negation:
   In a sentence like "I am not happy", the word "not" is very important.
   If it is removed, the sentence becomes "I am happy", which completely changes the meaning.

2. Question answering or search systems:
   In a query like "What is the capital of India?", words like "is" and "of" help maintain the structure of the sentence.
   Removing them may lead to incorrect interpretation and poor results.

---

## Q4. Difference between Stemming and Lemmatization

Stemming:
- Removes word endings using simple rules
- Output may not be a real word
- Example: "studies" becomes "studi"
- Faster but less accurate

Lemmatization:
- Converts words to their base or dictionary form
- Output is always a valid word
- Example: "studies" becomes "study"
- Slower but more accurate

In simple terms, stemming is faster but less precise, while lemmatization is slower but gives better results.

---
##  Setup: Install Required Libraries

In [17]:
# Install required libraries (run once)
!pip install nltk

In [18]:
# Import all required libraries
import re
import string
from collections import Counter
import nltk

# Download NLTK resources (requires internet connection)
NLTK_AVAILABLE = False
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    from nltk.corpus import stopwords as nltk_stopwords
    _ = nltk_stopwords.words('english')  # Force-load to detect missing data
    NLTK_AVAILABLE = True
    print(" NLTK stopwords loaded successfully.")
except Exception:
    NLTK_AVAILABLE = False
    print("  NLTK unavailable – using built-in stopword list as fallback.")

# Built-in English stopword list (used as fallback or always available)
BUILTIN_STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you',
    "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself',
    'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her',
    'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom',
    'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was',
    'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
    'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
    'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with',
    'about', 'against', 'between', 'into', 'through', 'during', 'before',
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once',
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'both', 'each',
    'few', 'more', 'most', 'other', 'some', 'such', 'own', 'same', 'so',
    'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don',
    "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're',
    've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn',
    "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't",
    'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn',
    "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't",
    'wouldn', "wouldn't", 'got', 'get', 'via', 'also', 'upon', 'call'
}

def get_stopwords():
    """Returns the best available stopword set."""
    if NLTK_AVAILABLE:
        return set(nltk_stopwords.words('english'))
    return BUILTIN_STOPWORDS

print(" All libraries imported successfully.")

 NLTK stopwords loaded successfully.
 All libraries imported successfully.


---
##  Task 2: Build Advanced Preprocessing Function

In [19]:
def preprocess_text(text):
    """
    Advanced NLP Preprocessing Function.

    Steps:
    1. Handle edge cases (None, non-string, empty, only emojis, only numbers)
    2. Remove URLs and email-like patterns
    3. Remove emojis and special characters
    4. Handle repeated characters (e.g., 'loooove' -> 'love')
    5. Convert to lowercase
    6. Remove numbers
    7. Remove punctuation
    8. Tokenize
    9. Remove very short tokens (length <= 2), keeping meaningful words like 'no', 'not'
    10. Remove stopwords (while preserving negations)
    11. Remove extra spaces

    Returns:
        tokens (list): List of cleaned tokens
        clean_sentence (str): Cleaned sentence
    """

    # ── Task 7: Error Handling ──────────────────────────────────────────────

    # Handle None or non-string input
    if text is None or not isinstance(text, str):
        return [], ""

    # Handle empty string
    if text.strip() == "":
        return [], "[EMPTY INPUT]"

    # Handle only-numbers input
    if re.fullmatch(r'[\d\s\+\-\.]+', text.strip()):
        return [], "[ONLY NUMBERS – NO MEANINGFUL TEXT]"

    # Handle only-emojis input (text with no ASCII letters)
    # Remove emojis and check if anything remains
    emoji_stripped = re.sub(
        r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF'
        r'\U00002702-\U000027B0\U000024C2-\U0001F251]+',
        '', text
    ).strip()
    if emoji_stripped == "":
        return [], "[ONLY EMOJIS – NO TEXT CONTENT]"

    # ── Step 1: Remove URLs and email-like patterns ─────────────────────────
    # Matches http/https URLs and www links
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Matches email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # ── Step 2: Remove emojis ───────────────────────────────────────────────
    text = re.sub(
        r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF'
        r'\U00002702-\U000027B0\U000024C2-\U0001F251]+',
        '', text
    )

    # ── Step 3: Handle repeated characters ─────────────────────────────────
    # E.g., 'loooove' -> 'love', 'soooo' -> 'so', 'goooood' -> 'good'
    # Replaces 3+ repeated characters with just 2 (then further reduced by regex)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Further reduce double characters that are not natural doubles
    # (Kept as 2 to retain words like 'good', 'feel', 'cool')

    # ── Step 4: Convert to lowercase ────────────────────────────────────────
    text = text.lower()

    # ── Step 5: Remove numbers ──────────────────────────────────────────────
    text = re.sub(r'\d+', '', text)

    # ── Step 6: Remove punctuation and special characters ───────────────────
    # Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # ── Step 7: Remove extra spaces ─────────────────────────────────────────
    text = re.sub(r'\s+', ' ', text).strip()

    # ── Step 8: Tokenize ────────────────────────────────────────────────────
    tokens = text.split()

    # ── Step 9: Define meaningful short words to preserve ───────────────────
    preserve_words = {'no', 'not', 'ok', 'go'}

    # ── Step 10: Remove very short tokens (length <= 2) unless meaningful ───
    tokens = [t for t in tokens if len(t) > 2 or t in preserve_words]

    # ── Step 11: Remove stopwords (preserve negations) ──────────────────────
    stop_words = get_stopwords()
    # Keep important negation words even if they are stopwords
    negation_words = {'not', 'no', 'nor', 'neither', 'never', 'none'}
    stop_words -= negation_words  # Remove negations from stopword list

    tokens = [t for t in tokens if t not in stop_words]

    # ── Build clean sentence ─────────────────────────────────────────────────
    clean_sentence = ' '.join(tokens)

    return tokens, clean_sentence


print(" preprocess_text() function defined successfully.")

# Quick sanity check
sample = "WOW!!! This is GREAT!!!"
tokens, sentence = preprocess_text(sample)
print(f"\nSanity Check:")
print(f"  Input   : {sample}")
print(f"  Tokens  : {tokens}")
print(f"  Cleaned : {sentence}")

 preprocess_text() function defined successfully.

Sanity Check:
  Input   : WOW!!! This is GREAT!!!
  Tokens  : ['wow', 'great']
  Cleaned : wow great


---
## 🧪 Task 3: Stress Testing
Testing the preprocessing function on 10 diverse real-world sentences.

In [20]:
# ── 10 Diverse Test Sentences ───────────────────────────────────────────────
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# ── Process Each Sentence and Display Results ────────────────────────────────
results = []  # Store for later analysis

print("=" * 65)
print("           STRESS TEST – NLP PREPROCESSING RESULTS")
print("=" * 65)

for i, sentence in enumerate(test_sentences, 1):
    tokens, clean_sent = preprocess_text(sentence)
    results.append({
        'original': sentence,
        'tokens': tokens,
        'clean_sentence': clean_sent
    })
    print(f"\n[Sentence {i}]")
    print(f"   Original Text  : {sentence}")
    print(f"   Cleaned Tokens : {tokens}")
    print(f"   Cleaned Sentence: {clean_sent}")
    print("-" * 65)

print("\n✅ Stress testing complete.")

           STRESS TEST – NLP PREPROCESSING RESULTS

[Sentence 1]
   Original Text  : Get 100% FREE access now!!!
   Cleaned Tokens : ['get', 'free', 'access']
   Cleaned Sentence: get free access
-----------------------------------------------------------------

[Sentence 2]
   Original Text  : I absolutely looooved this product 😍😍
   Cleaned Tokens : ['absolutely', 'looved', 'product']
   Cleaned Sentence: absolutely looved product
-----------------------------------------------------------------

[Sentence 3]
   Original Text  : Worst service ever... 0/10
   Cleaned Tokens : ['worst', 'service', 'ever']
   Cleaned Sentence: worst service ever
-----------------------------------------------------------------

[Sentence 4]
   Original Text  : Call me at 9876543210
   Cleaned Tokens : ['call']
   Cleaned Sentence: call
-----------------------------------------------------------------

[Sentence 5]
   Original Text  : This is THE best course!!!
   Cleaned Tokens : ['best', 'course']
   C

---
##  Task 4: Token Analytics

In [21]:
print("=" * 65)
print("                   TOKEN ANALYTICS")
print("=" * 65)

noise_scores = []    # To find sentence with most noise
retained_scores = [] # To find sentence retaining most meaningful tokens

for i, res in enumerate(results, 1):
    original_word_count = len(res['original'].split())
    total_tokens = len(res['tokens'])
    unique_tokens = len(set(res['tokens']))
    avg_token_len = (
        round(sum(len(t) for t in res['tokens']) / total_tokens, 2)
        if total_tokens > 0 else 0
    )
    noise = original_word_count - total_tokens  # Tokens removed = noise

    noise_scores.append((i, noise, res['original']))
    retained_scores.append((i, total_tokens, res['original']))

    print(f"\n[Sentence {i}]: \"{res['original'][:45]}...\"" if len(res['original']) > 45
          else f"\n[Sentence {i}]: \"{res['original']}\"")
    print(f"  Total Tokens   : {total_tokens}")
    print(f"  Unique Tokens  : {unique_tokens}")
    print(f"  Avg Token Len  : {avg_token_len} characters")

# ── Analysis Questions ───────────────────────────────────────────────────────
most_noisy = max(noise_scores, key=lambda x: x[1])
most_retained = max(retained_scores, key=lambda x: x[1])

print("\n" + "=" * 65)
print(" ANALYSIS QUESTIONS")
print("=" * 65)
print(f"\n Most Noisy Sentence (Sentence {most_noisy[0]}):")
print(f"   '{most_noisy[2]}'")
print(f"   → {most_noisy[1]} tokens were removed as noise")

print(f"\n Most Meaningful Tokens Retained (Sentence {most_retained[0]}):")
print(f"   '{most_retained[2]}'")
print(f"   → {most_retained[1]} meaningful tokens retained after cleaning")

                   TOKEN ANALYTICS

[Sentence 1]: "Get 100% FREE access now!!!"
  Total Tokens   : 3
  Unique Tokens  : 3
  Avg Token Len  : 4.33 characters

[Sentence 2]: "I absolutely looooved this product 😍😍"
  Total Tokens   : 3
  Unique Tokens  : 3
  Avg Token Len  : 7.67 characters

[Sentence 3]: "Worst service ever... 0/10"
  Total Tokens   : 3
  Unique Tokens  : 3
  Avg Token Len  : 5.33 characters

[Sentence 4]: "Call me at 9876543210"
  Total Tokens   : 1
  Unique Tokens  : 1
  Avg Token Len  : 4.0 characters

[Sentence 5]: "This is THE best course!!!"
  Total Tokens   : 2
  Unique Tokens  : 2
  Avg Token Len  : 5.0 characters

[Sentence 6]: "Visit https://openai.com now!"
  Total Tokens   : 1
  Unique Tokens  : 1
  Avg Token Len  : 5.0 characters

[Sentence 7]: "Nooooo this is baaad!!!"
  Total Tokens   : 2
  Unique Tokens  : 2
  Avg Token Len  : 3.5 characters

[Sentence 8]: "OK OK OK I got it"
  Total Tokens   : 4
  Unique Tokens  : 2
  Avg Token Len  : 2.25 characters

[S

---
##  Task 5: Frequency Analysis

In [22]:
from collections import Counter

# Combine all tokens from all processed sentences
all_tokens = []
for res in results:
    all_tokens.extend(res['tokens'])

print(f"Total tokens across all sentences: {len(all_tokens)}")
print(f"Total unique tokens               : {len(set(all_tokens))}")

# Frequency count
token_freq = Counter(all_tokens)

# ── Top 10 Most Frequent Words ───────────────────────────────────────────────
print("\n" + "=" * 45)
print("    Top 10 Most Frequent Words")
print("=" * 45)
print(f"{'Rank':<6} {'Word':<20} {'Frequency':<10}")
print("-" * 45)
for rank, (word, freq) in enumerate(token_freq.most_common(10), 1):
    print(f"{rank:<6} {word:<20} {freq:<10}")

# ── Top 5 Least Frequent Words ───────────────────────────────────────────────
print("\n" + "=" * 45)
print("    Top 5 Least Frequent Words")
print("=" * 45)
print(f"{'Rank':<6} {'Word':<20} {'Frequency':<10}")
print("-" * 45)
least_common = token_freq.most_common()[:-6:-1]  # Last 5
for rank, (word, freq) in enumerate(least_common, 1):
    print(f"{rank:<6} {word:<20} {freq:<10}")

Total tokens across all sentences: 24
Total unique tokens               : 22

    Top 10 Most Frequent Words
Rank   Word                 Frequency 
---------------------------------------------
1      ok                   3         
2      get                  1         
3      free                 1         
4      access               1         
5      absolutely           1         
6      looved               1         
7      product              1         
8      worst                1         
9      service              1         
10     ever                 1         

    Top 5 Least Frequent Words
Rank   Word                 Frequency 
---------------------------------------------
1      happy                1         
2      not                  1         
3      offer                1         
4      limited              1         
5      win                  1         


---
##  Task 6: Build Full Pipeline

In [23]:
def full_pipeline(text_list):
    """
    Full NLP Preprocessing Pipeline.

    Accepts a list of raw text strings and returns a dictionary containing:
    - 'tokens': list of token lists (one per sentence)
    - 'clean_sentences': list of cleaned sentences

    Args:
        text_list (list): List of raw input strings

    Returns:
        dict: {
            'tokens': [[...], [...], ...],
            'clean_sentences': ['...', '...', ...]
        }
    """

    # Input validation
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")

    if len(text_list) == 0:
        return {"tokens": [], "clean_sentences": []}

    all_tokens = []
    all_clean_sentences = []

    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens.append(tokens)
        all_clean_sentences.append(clean_sentence)

    return {
        "tokens": all_tokens,
        "clean_sentences": all_clean_sentences
    }


print(" full_pipeline() function defined.")

# ── Run the Full Pipeline on Test Data ──────────────────────────────────────
pipeline_output = full_pipeline(test_sentences)

print("\n" + "=" * 65)
print("           FULL PIPELINE OUTPUT")
print("=" * 65)

for i, (tokens, clean) in enumerate(
    zip(pipeline_output['tokens'], pipeline_output['clean_sentences']), 1
):
    print(f"\n[{i}] Tokens         : {tokens}")
    print(f"    Clean Sentence : {clean}")

 full_pipeline() function defined.

           FULL PIPELINE OUTPUT

[1] Tokens         : ['get', 'free', 'access']
    Clean Sentence : get free access

[2] Tokens         : ['absolutely', 'looved', 'product']
    Clean Sentence : absolutely looved product

[3] Tokens         : ['worst', 'service', 'ever']
    Clean Sentence : worst service ever

[4] Tokens         : ['call']
    Clean Sentence : call

[5] Tokens         : ['best', 'course']
    Clean Sentence : best course

[6] Tokens         : ['visit']
    Clean Sentence : visit

[7] Tokens         : ['noo', 'baad']
    Clean Sentence : noo baad

[8] Tokens         : ['ok', 'ok', 'ok', 'got']
    Clean Sentence : ok ok ok got

[9] Tokens         : ['win', 'limited', 'offer']
    Clean Sentence : win limited offer

[10] Tokens         : ['not', 'happy']
    Clean Sentence : not happy


---
## Task 7: Error Handling
Testing edge cases: empty string, only emojis, only numbers.

In [24]:
# ── Edge Case Tests ──────────────────────────────────────────────────────────
edge_cases = [
    ("",              "Empty String"),
    ("😍😂🔥💯",       "Only Emojis"),
    ("1234567890",    "Only Numbers"),
    (None,            "None Input"),
    ("   ",           "Only Whitespace"),
]

print("=" * 65)
print("           ERROR HANDLING – EDGE CASE RESULTS")
print("=" * 65)

for input_val, label in edge_cases:
    tokens, clean_sent = preprocess_text(input_val)
    print(f"\n🔹 Case     : {label}")
    print(f"   Input    : {repr(input_val)}")
    print(f"   Tokens   : {tokens}")
    print(f"   Output   : {repr(clean_sent)}")
    print("-" * 65)

print("\n All edge cases handled gracefully.")

           ERROR HANDLING – EDGE CASE RESULTS

🔹 Case     : Empty String
   Input    : ''
   Tokens   : []
   Output   : '[EMPTY INPUT]'
-----------------------------------------------------------------

🔹 Case     : Only Emojis
   Input    : '😍😂🔥💯'
   Tokens   : []
   Output   : '[ONLY EMOJIS – NO TEXT CONTENT]'
-----------------------------------------------------------------

🔹 Case     : Only Numbers
   Input    : '1234567890'
   Tokens   : []
   Output   : '[ONLY NUMBERS – NO MEANINGFUL TEXT]'
-----------------------------------------------------------------

🔹 Case     : None Input
   Input    : None
   Tokens   : []
   Output   : ''
-----------------------------------------------------------------

🔹 Case     : Only Whitespace
   Input    : '   '
   Tokens   : []
   Output   : '[EMPTY INPUT]'
-----------------------------------------------------------------

 All edge cases handled gracefully.
